# Train GCN Model

In [1]:
from IPython.display import display
import os

if "SSH_CONNECTION" in os.environ:
    display("Running via SSH")
else:
    display("Running locally")
    
import sys
import os

path = os.path.join('..', '.')
if path not in sys.path:
    sys.path.append(os.path.abspath(path))

import random

import numpy as np
import pandas as pd
import pickle as pkl

import torch
from torch_geometric.data import Data

import warnings
warnings.filterwarnings('ignore')

from src import run_model, protein_graph, gcn_model, evaluation


torch.cuda.is_available()

'Running via SSH'

True

In [2]:
# dataset_path = 'datasets/x18_exp_l2_graph_dict.pkl'
# dataset_path = 'datasets/clustered_2_graph_dict.pkl'
dataset_path = 'datasets/codon_graph_dict.pkl'
with open(dataset_path, 'rb') as f:
    graph_dict = pkl.load(f)
    
print(len(graph_dict['train']) + len(graph_dict['test']))

664


### Train

- Cutoff distance = 12
- Dropout = 0.6
- Edge weights = "exp"
- Edge weight lambda = 2
- Hidden channels = 256
- Learning rate = 4.5e-5
- Weight decay = 5e-6


In [3]:
seed = 42
np.random.seed(seed)
random.seed(seed)

# logging params (only used for wandb metrics)
n_samples = len(graph_dict['train']) + len(graph_dict['test'])
cutoff_distance = 12

# gcn params
num_node_features = 18
batch_size = 256
hidden_channels = 256
dropout = 0.6

edge_weight_func = "exp"
edge_weight_lambda = 2

learning_rate = 4e-5
wd = 5e-6
# epochs = 1119
epochs = 1800

In [4]:
project = 'pnca-clustered-split'
run_name = 'codon-split-long'
# run_name = 'clustered_2-split-1'

In [5]:
model = run_model.pnca_GCN_vary_graph(
            self_loops = False,
            cutoff_distance = cutoff_distance,
            edge_weight_func = edge_weight_func,
            batch_size = batch_size,
            num_node_features = num_node_features,
            hidden_channels = hidden_channels,
            learning_rate = learning_rate,
            wd = wd,
            dropout = dropout,
            lr_scheduling=False,
            epochs = epochs,
            graph_dict= graph_dict,
            normalise_ews=True,
            lambda_param= edge_weight_lambda,
            early_stop=False,
            # save_path= f'../saved_models/{project}/{run_name}',
            wandb_params={
              'use_wandb': True, 
              'wandb_project': f'{project}', 
              'wandb_name': f'{run_name}',
              'n_samples': n_samples,
              'sweep': False
              }
        )

Using CUDA


Failed to detect the name of this notebook, you can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
/usr/lib/python3/dist-packages/requests/__init__.py:87: RequestsDependencyWarning: urllib3 (2.2.3) or chardet (4.0.0) doesn't match a supported version!
  warnings.warn("urllib3 ({}) or chardet ({}) doesn't match a supported "
wandb: Currently logged in as: dylan-home. Use `wandb login --relogin` to force relogin


Epoch: 000, Train Acc: 0.4914, Test Acc: 0.4350, Train Loss: 0.6945, Test Loss: 0.6982
Epoch: 010, Train Acc: 0.4914, Test Acc: 0.4350, Train Loss: 0.6936, Test Loss: 0.6963
Epoch: 020, Train Acc: 0.5022, Test Acc: 0.5400, Train Loss: 0.6922, Test Loss: 0.6907
Epoch: 030, Train Acc: 0.5129, Test Acc: 0.5500, Train Loss: 0.6913, Test Loss: 0.6898
Epoch: 040, Train Acc: 0.5129, Test Acc: 0.5450, Train Loss: 0.6907, Test Loss: 0.6888
Epoch: 050, Train Acc: 0.5216, Test Acc: 0.5550, Train Loss: 0.6905, Test Loss: 0.6866
Epoch: 060, Train Acc: 0.5216, Test Acc: 0.5400, Train Loss: 0.6895, Test Loss: 0.6862
Epoch: 070, Train Acc: 0.5216, Test Acc: 0.5450, Train Loss: 0.6890, Test Loss: 0.6861
Epoch: 080, Train Acc: 0.5280, Test Acc: 0.5550, Train Loss: 0.6882, Test Loss: 0.6859
Epoch: 090, Train Acc: 0.5366, Test Acc: 0.5600, Train Loss: 0.6880, Test Loss: 0.6842
Epoch: 100, Train Acc: 0.5517, Test Acc: 0.5550, Train Loss: 0.6870, Test Loss: 0.6832
Epoch: 110, Train Acc: 0.5496, Test Acc: 0.

wandb: WARNING Source type is set to 'repo' but some required information is missing from the environment. A job will not be created from this run. See https://docs.wandb.ai/guides/launch/create-job


Test Accuracy,▁▃▃▄▅▆▆▆▇▇▇▇▇▇▇▇▇████████████▇████████▇█
Test F1,▁▇▇▇▇▇▇▇▇▇▇▇▇▇█▇█████████████▇████████▇█
Test Loss,████▇▇▆▅▄▄▃▃▃▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▃▁▂▁▁▁▁▁▂▄▂
Test Sensitivity,▁▇▇██▇▆▆▆▆▇▆▆▆▇▇▇▇▇▇█▇▇▇▇█▇██▆▇▇▇▇▇▇▇▇▆▇
Test Specificity,█▁▂▂▂▅▆▆▇▇▆▇▇▇▆▇▇▇▇▇▆▇▆▇▇▆▇▆▆▇▆▇▆▆▆▆▆▇▇▆
Train Accuracy,▁▁▂▃▄▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇▇▇▇█▇▇███▇█▇██████▇█
Train F1,▁▆▆▆▇▇▇▇▇▇▇▇▇▇▇▇▇████████████▇██████████
Train Loss,█████▇▆▆▅▅▅▄▄▄▄▃▃▃▃▃▃▃▃▃▃▂▂▂▂▃▂▂▂▂▁▁▁▁▂▁
Train Sensitivity,▁█▇▇█▇▆▆▆▆▇▆▇▇▇▇▇▇▇▇█▇▇▇▇█▇▇▇▆▇▇▇▇███▇▆▇
Train Specificity,█▁▂▂▃▅▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇█▇█▇▇▇▇▇▇█▇
Test Accuracy,0.8
